In [2]:
!pip install segmentation-models-pytorch opencv-python scikit-learn
import os
import json
import random
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import segmentation_models_pytorch as smp
import torchvision.transforms as transforms
from sklearn.metrics import roc_auc_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.7 MB/s eta 0:00:00


In [5]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"  # set to "1" only when debugging CUDA errors
torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [6]:
DATASET_PATH = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset"

TRAIN_IMG = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/train/image"
TRAIN_ANNO = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/train/annos"

VAL_IMG = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/validation/image"
VAL_ANNO = "/kaggle/input/datasets/shivvvm/filtered-dataset-v1/filtered_dataset/validation/annos"


# print(len(os.listdir(IMAGE_DIR)))

In [7]:
rng = random.Random(SEED)

train_images_all = sorted(os.listdir(TRAIN_IMG))
val_images_all = sorted(os.listdir(VAL_IMG))
rng.shuffle(train_images_all)
rng.shuffle(val_images_all)

train_images = train_images_all[:30000]
val_images = val_images_all[:5000]

print(len(train_images), len(val_images))

30000 5000


In [9]:
NUM_CLASSES = 5
CLASS_MAP = {1:0, 2:1, 7:2, 8:3, 9:4} 

In [10]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

cuda


In [12]:
class FashionSegDataset(Dataset):

    def __init__(self, img_dir, anno_dir, files):
        self.img_dir = img_dir
        self.anno_dir = anno_dir
        self.files = files

        # define transform once (better practice)
        self.transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(
                [0.485,0.456,0.406],
                [0.229,0.224,0.225]
            )
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        img_name = self.files[idx]

        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        w, h = image.size
        mask = Image.new("L", (w, h), 0)

        anno_file = img_name.replace(".jpg", ".json")
        anno_path = os.path.join(self.anno_dir, anno_file)

        with open(anno_path) as f:
            data = json.load(f)

        draw = ImageDraw.Draw(mask)

        for item in data.values():

            cid = item["category_id"]

            if cid in CLASS_MAP:

                label = CLASS_MAP[cid] + 1   # background = 0

                seg = item["segmentation"][0]
                polygon = [(seg[i], seg[i+1]) for i in range(0,len(seg),2)]

                draw.polygon(polygon, fill=label)

        # apply transforms
        image = self.transform(image)

        mask = transforms.Resize(
            (224,224),
            interpolation=Image.NEAREST
        )(mask)

        mask = torch.from_numpy(np.array(mask)).long()

        return image, mask

In [14]:
# Augmented dataset (geometric + color jitter) with paired transforms on image/mask
import torchvision.transforms.functional as F

class AugmentedFashionSegDataset(Dataset):
    def __init__(self, img_dir, anno_dir, files, augment=False):
        self.img_dir = img_dir
        self.anno_dir = anno_dir
        self.files = files
        self.augment = augment

        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.resize_mask = transforms.Resize((224, 224), interpolation=Image.NEAREST)
        self.color_aug = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        w, h = image.size
        mask = Image.new("L", (w, h), 0)
        anno_file = img_name.replace(".jpg", ".json")
        anno_path = os.path.join(self.anno_dir, anno_file)

        data = {}
        if os.path.exists(anno_path):
            try:
                with open(anno_path) as f:
                    data = json.load(f)
            except Exception:
                data = {}

        draw = ImageDraw.Draw(mask)
        for item in data.values():
            cid = item.get("category_id")
            if cid in CLASS_MAP:
                segs = item.get("segmentation", [])
                if not segs or not isinstance(segs[0], list):
                    continue
                seg = segs[0]
                if len(seg) < 6 or len(seg) % 2 != 0:
                    continue
                label = CLASS_MAP[cid] + 1
                polygon = [(seg[i], seg[i + 1]) for i in range(0, len(seg), 2)]
                draw.polygon(polygon, fill=label)

        # augmentations (geom shared, color on image only)
        if self.augment:
            if random.random() < 0.5:
                image = F.hflip(image)
                mask = F.hflip(mask)
            if random.random() < 0.5:
                image = F.vflip(image)
                mask = F.vflip(mask)
            angle = random.uniform(-15, 15)
            image = F.rotate(image, angle, interpolation=F.InterpolationMode.BILINEAR)
            mask = F.rotate(mask, angle, interpolation=F.InterpolationMode.NEAREST)
            image = self.color_aug(image)

        # resize + normalize
        image = F.resize(image, (224, 224), interpolation=F.InterpolationMode.BILINEAR)
        image = self.normalize(image)
        mask = self.resize_mask(mask)
        mask = torch.from_numpy(np.array(mask)).long()
        return image, mask

In [ ]:
# Rebind datasets/loaders with augmentation for training only
train_dataset = AugmentedFashionSegDataset(TRAIN_IMG, TRAIN_ANNO, train_images, augment=True)
val_dataset = AugmentedFashionSegDataset(VAL_IMG, VAL_ANNO, val_images, augment=False)

pin_mem = device.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,         
    pin_memory=pin_mem,
    persistent_workers=False, 
    prefetch_factor=2,
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=pin_mem,
    persistent_workers=False,
    prefetch_factor=2,
    drop_last=False,
)

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet34",        
    encoder_weights="imagenet",     
    in_channels=3,
    classes=NUM_CLASSES + 1         
).to(device)


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [18]:
class DiceLoss(nn.Module):
    def forward(self, preds, targets):

        preds = torch.softmax(preds, dim=1)

        targets_one_hot = torch.nn.functional.one_hot(
            targets, num_classes=NUM_CLASSES+1
        ).permute(0,3,1,2).float()

        preds = preds[:,1:]
        targets_one_hot = targets_one_hot[:,1:]

        intersection = (preds * targets_one_hot).sum(dim=(2,3))
        union = preds.sum(dim=(2,3)) + targets_one_hot.sum(dim=(2,3))

        dice = (2 * intersection + 1e-8) / (union + 1e-8)

        return 1 - dice.mean()

In [19]:
# Compute class weights to handle imbalance (background down-weighted)
def compute_class_weights(max_samples=2000):
    counts = np.ones(NUM_CLASSES + 1, dtype=np.float64)  # start with 1 to avoid zero-divide
    sample_files = train_images[:max_samples]
    for img_name in sample_files:
        img_path = os.path.join(TRAIN_IMG, img_name)
        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        mask = Image.new("L", (w, h), 0)
        anno_file = img_name.replace(".jpg", ".json")
        anno_path = os.path.join(TRAIN_ANNO, anno_file)
        with open(anno_path) as f:
            data = json.load(f)
        draw = ImageDraw.Draw(mask)
        for item in data.values():
            cid = item["category_id"]
            if cid in CLASS_MAP:
                label = CLASS_MAP[cid] + 1
                seg = item["segmentation"][0]
                polygon = [(seg[i], seg[i+1]) for i in range(0, len(seg), 2)]
                draw.polygon(polygon, fill=label)
        mask_np = np.array(mask, dtype=np.int64)
        binc = np.bincount(mask_np.flatten(), minlength=NUM_CLASSES + 1)
        counts[: len(binc)] += binc
    # inverse frequency weights
    weights = counts.max() / (counts + 1e-6)
    # reduce background weight
    weights[0] = weights[1:].min() * 0.5
    # normalize to mean=1 for stability
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=device)

class_weights = compute_class_weights()
print("Class weights:", class_weights.cpu().numpy())

Class weights: [0.26356342 0.52712685 0.963882   1.8414605  0.8917297  1.5122375 ]


In [20]:
class CombinedLoss(nn.Module):
    def __init__(self, class_weights):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss()

    def forward(self, preds, targets):
        return self.ce(preds, targets) + self.dice(preds, targets)

criterion = CombinedLoss(class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5, weight_decay=1e-5)

In [21]:
def seg_metrics(out, mask):
    pred = torch.argmax(out, 1).cpu().numpy()
    mask = mask.cpu().numpy()

    iou_list, dice_list = [], []

    for c in range(1, NUM_CLASSES + 1):
        p = pred == c
        m = mask == c

        inter = (p & m).sum()
        union = (p | m).sum()

        if union == 0:
            continue

        iou_list.append(inter / union)
        dice_list.append((2 * inter) / (p.sum() + m.sum() + 1e-8))

    if not iou_list:
        return 0.0, 0.0

    return float(np.mean(iou_list)), float(np.mean(dice_list))

In [22]:
def compute_auc(outputs, masks):

    probs = torch.softmax(outputs, dim=1).cpu().numpy()
    masks = masks.cpu().numpy()

    aucs = []

    for c in range(1, NUM_CLASSES+1):

        gt = (masks == c).astype(int).flatten()
        pred = probs[:,c,:,:].flatten()

        if np.sum(gt)==0: continue

        try:
            auc = roc_auc_score(gt, pred)
            aucs.append(auc)
        except:
            continue

    return np.mean(aucs) if aucs else 0

In [25]:
def bbox_iou(a,b):
    xA,yA = max(a[0],b[0]), max(a[1],b[1])
    xB,yB = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0,xB-xA)*max(0,yB-yA)
    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])
    union = areaA+areaB-inter
    return inter/union if union>0 else 0

In [26]:
def mask_to_boxes(mask):
    boxes=[]
    for c in range(1,NUM_CLASSES+1):
        binary=(mask==c).astype(np.uint8)
        contours,_=cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            x,y,w,h=cv2.boundingRect(cnt)
            boxes.append((c,[x,y,x+w,y+h]))
    return boxes

In [ ]:
def compute_box_precision(preds, gts):
    thresholds = np.arange(0.5, 1.0, 0.05)
    precisions = []

    if len(preds) == 0 and len(gts) == 0:
        return 1.0, 1.0
    if len(preds) == 0:
        return 0.0, 0.0

    for t in thresholds:
        tp = 0
        fp = 0
        matched = set()

        for p in preds:
            found = False
            for i, g in enumerate(gts):
                if i in matched:
                    continue
                if p[0] == g[0] and bbox_iou(p[1], g[1]) >= t:
                    tp += 1
                    matched.add(i)
                    found = True
                    break
            if not found:
                fp += 1

        precisions.append(tp / (tp + fp + 1e-8))

    map_50 = precisions[0]
    map_50_95 = np.mean(precisions)
    return float(map_50), float(map_50_95)

In [ ]:
import math
import numpy as np
import torch
from torch.amp import autocast, GradScaler
from sklearn.metrics import precision_recall_fscore_support

# ------------------ CONFIG ------------------
num_epochs = 50
patience = 10
save_path = "unet_best.pth"

best_iou = -math.inf
patience_ctr = 0

device_type = "cuda" if device.type == "cuda" else "cpu"

# Scheduler (NO autocast here )
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs
)

# GradScaler (FIXED)
scaler = GradScaler(device=device_type, enabled=(device.type == "cuda"))

# ------------------ TRAIN LOOP ------------------
for epoch in range(1, num_epochs + 1):

    # -------- TRAIN --------
    model.train()
    train_loss = 0.0

    for imgs, masks in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # AMP ONLY here
        with autocast(device_type=device_type, enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()

    # -------- VALIDATION --------
    model.eval()

    val_loss = 0.0
    iou_scores, dice_scores = [], []
    precision_list, recall_list, f1_list = [], [], []
    map50_list, map50_95_list = [], []

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with autocast(device_type=device_type, enabled=(device.type == "cuda")):
                logits = model(imgs)
                loss = criterion(logits, masks)

            val_loss += loss.item()

            # ---- Segmentation metrics ----
            iou, dice = seg_metrics(logits, masks)
            iou_scores.append(iou)
            dice_scores.append(dice)

            # ---- Predictions ----
            preds = torch.argmax(logits, dim=1)

            preds_flat = preds.cpu().numpy().reshape(-1)
            gts_flat = masks.cpu().numpy().reshape(-1)

            # ---- Classification metrics ----
            precision_val, recall_val, f1_val, _ = precision_recall_fscore_support(
                gts_flat,
                preds_flat,
                labels=list(range(1, NUM_CLASSES + 1)),
                average="weighted",
                zero_division=0,
            )

            precision_list.append(precision_val)
            recall_list.append(recall_val)
            f1_list.append(f1_val)

            # ---- Box precision ----
            for b in range(imgs.size(0)):
                pred_mask = preds[b].cpu().numpy()
                gt_mask = masks[b].cpu().numpy()

                pred_boxes = mask_to_boxes(pred_mask)
                gt_boxes = mask_to_boxes(gt_mask)

                map_50, map_50_95 = compute_box_precision(pred_boxes, gt_boxes)
                map50_list.append(map_50)
                map50_95_list.append(map_50_95)

    # -------- AGGREGATE --------
    val_loss /= len(val_loader)

    mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
    mean_dice = float(np.mean(dice_scores)) if dice_scores else 0.0
    mean_precision = float(np.mean(precision_list)) if precision_list else 0.0
    mean_recall = float(np.mean(recall_list)) if recall_list else 0.0
    mean_f1 = float(np.mean(f1_list)) if f1_list else 0.0
    mean_map_50 = float(np.mean(map50_list)) if map50_list else 0.0
    mean_map_50_95 = float(np.mean(map50_95_list)) if map50_95_list else 0.0

    # -------- LOG --------
    print(
        f"Epoch {epoch}/{num_epochs} | "
        f"train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | "
        f"IoU: {mean_iou:.3f} | Dice: {mean_dice:.3f} | "
        f"P: {mean_precision:.3f} R: {mean_recall:.3f} F1: {mean_f1:.3f} | "
        f"mAP@0.5: {mean_map_50:.3f} mAP@0.5:0.95: {mean_map_50_95:.3f}"
    )

    # -------- EARLY STOPPING --------
    if mean_iou > best_iou:
        best_iou = mean_iou
        patience_ctr = 0

        torch.save(model.state_dict(), save_path)
        print(f"New best model saved to {save_path}")

    else:
        patience_ctr += 1
        print(f"No improvement. Patience: {patience_ctr}/{patience}")

        if patience_ctr >= patience:
            print("Early stopping triggered.")
            break

# ------------------ DONE ------------------
print(f"\nBest validation mIoU: {best_iou:.4f}")

Epoch 1/50 | train_loss: 2.3740 | val_loss: 2.2221 | IoU: 0.070 | Dice: 0.110 | P: 0.315 R: 0.172 F1: 0.177 | BoxPrec: 0.023
✅ New best model saved to unet_best.pth


KeyboardInterrupt: 

Exception in thread Thread-7 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_s

In [ ]:
# Evaluate saved best model on validation set
model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

val_loss = 0.0
iou_scores, dice_scores = [], []
precision_list, recall_list, f1_list = [], [], []
map50_list, map50_95_list = [], []

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        with autocast(device_type="cuda" if device.type == "cuda" else "cpu", enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss = criterion(logits, masks)

        val_loss += loss.item()

        iou, dice = seg_metrics(logits, masks)
        iou_scores.append(iou)
        dice_scores.append(dice)

        preds_flat = torch.argmax(logits, 1).cpu().numpy().reshape(-1)
        gts_flat = masks.cpu().numpy().reshape(-1)
        precision_val, recall_val, f1_val, _ = precision_recall_fscore_support(
            gts_flat,
            preds_flat,
            labels=list(range(1, NUM_CLASSES + 1)),
            average="weighted",
            zero_division=0,
        )
        precision_list.append(precision_val)
        recall_list.append(recall_val)
        f1_list.append(f1_val)

        for b in range(imgs.size(0)):
            pred_mask = torch.argmax(logits[b], 0).cpu().numpy()
            gt_mask = masks[b].cpu().numpy()
            pred_boxes = mask_to_boxes(pred_mask)
            gt_boxes = mask_to_boxes(gt_mask)
            map_50, map_50_95 = compute_box_precision(pred_boxes, gt_boxes)
            map50_list.append(map_50)
            map50_95_list.append(map_50_95)

val_loss /= len(val_loader)
mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
mean_dice = float(np.mean(dice_scores)) if dice_scores else 0.0
mean_precision = float(np.mean(precision_list)) if precision_list else 0.0
mean_recall = float(np.mean(recall_list)) if recall_list else 0.0
mean_f1 = float(np.mean(f1_list)) if f1_list else 0.0
mean_map_50 = float(np.mean(map50_list)) if map50_list else 0.0
mean_map_50_95 = float(np.mean(map50_95_list)) if map50_95_list else 0.0

print(
    f"[Eval] val_loss: {val_loss:.4f} | IoU: {mean_iou:.3f} | Dice: {mean_dice:.3f} | "
    f"P: {mean_precision:.3f} R: {mean_recall:.3f} F1: {mean_f1:.3f} | "
    f"mAP@0.5: {mean_map_50:.3f} mAP@0.5:0.95: {mean_map_50_95:.3f}"
)